# Multi-Profile Chemical Equilibrium: ML vs ExoGibbs

This notebook compares the chemical mixing ratios of 6 key species (**He, CO, H2, H2O, CH4, NH3**) across three P-T profiles.

**Models Compared:**
1. **ExoGibbs (AAS)** (Solid): Native JAX using Asplund solar abundances.
2. **ExoGibbs (Custom)** (Dotted): Native JAX using the ML model's specific training abundances.
3. **ML** (Dashed): The Transformer emulator results.

In [ ]:
import time
import sys
import types
from pathlib import Path

import numpy as np
import jax.numpy as jnp
from jax import config
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

config.update("jax_enable_x64", True)

def load_style():
    if Path("science.mplstyle").exists():
        plt.style.use("science.mplstyle")
    else:
        plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.3,
                             'xtick.direction': 'in', 'ytick.direction': 'in'})

def load_model_standalone(bundle_path):
    with np.load(bundle_path, allow_pickle=False) as f:
        src = bytes(f["meta/vulcan_emulator_src"]).decode()
    module = types.ModuleType("_embedded_vulcan_inference")
    sys.modules["_embedded_vulcan_inference"] = module
    exec(compile(src, "<embedded>", "exec"), module.__dict__)
    return module.load_model(bundle_path)

load_style()
bundle = load_model_standalone("best_exported.npz")
species_labels = bundle.species
print(f"ML model loaded. Species: {len(species_labels)}")


In [ ]:
import jax
print(jax.devices())
print(jax.default_backend())

In [ ]:
from exogibbs.presets.fastchem import chemsetup
from exogibbs.api.equilibrium import EquilibriumOptions, equilibrium_profile
from exojax.utils.zsol import nsol

chem = chemsetup()
opts = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")

# The ML model's training abundances (Custom)
ML_SOLAR_ABUNDANCES = {
    "He_H": 8.38e-2, "C_H": 2.95e-4, "O_H": 5.37e-4, "N_H": 7.08e-5, "S_H": 1.41e-5,
}

solar_exojax = nsol() # AAS Asplund defaults
element_vector_aas = jnp.append(jnp.array([solar_exojax[el] for el in chem.elements[:-1]]), 0.0)

my_solar = solar_exojax.copy()
my_solar.update({"He": 8.38e-2, "C": 2.95e-4, "O": 5.37e-4, "N": 7.08e-5, "S": 1.41e-5})
element_vector_my = jnp.append(jnp.array([my_solar[el] for el in chem.elements[:-1]]), 0.0)

print("ExoGibbs configured.")

In [ ]:
print("Warming up JAX...")
dp, dt = np.logspace(2, -7, 50), 1200.0 * (np.logspace(2, -7, 50))**0.1
_ = bundle.predict_fastchem(pressure_bar=dp, temperature_k=dt, global_inputs=ML_SOLAR_ABUNDANCES, return_log10=False)
_ = equilibrium_profile(chem, dt, dp, element_vector_aas, Pref=1.0, initializer=None, options=opts)
print("Warmup complete.")

In [ ]:
pressures = np.logspace(2, -7, 50)
pt_profiles = [
    ("Hot Power-Law", 2000.0 * (pressures)**0.05),
    ("Warm Power-Law", 1200.0 * (pressures)**0.1),
    ("Cool Isothermal", 800.0 * np.ones_like(pressures)),
]

species_map = [
    ("He",  ["He", "He1"], "tab:cyan"),
    ("CO",  ["C1O1", "CO"], "tab:blue"),
    ("H2",  ["H2"], "tab:orange"),
    ("H2O", ["H2O1", "O1H2", "H2O"], "tab:green"),
    ("CH4", ["C1H4", "H4C1", "CH4"], "tab:red"),
    ("NH3", ["N1H3", "H3N1", "NH3"], "tab:purple")
]

fig, axes = plt.subplots(3, 2, figsize=(14, 15), sharey='row', 
                       gridspec_kw={'width_ratios': [1, 2.5], 'wspace': 0.0, 'hspace': 0.3})

for i, (label, T) in enumerate(pt_profiles):
    # Timings and Inference
    t0 = time.time()
    p_ml = np.asarray(bundle.predict_fastchem(pressure_bar=pressures, temperature_k=T, 
                                             global_inputs=ML_SOLAR_ABUNDANCES, return_log10=False))
    dt_ml = (time.time() - t0) * 1000
    
    t0 = time.time()
    p_g_my = np.asarray(equilibrium_profile(chem, T, pressures, element_vector_my, Pref=1.0, options=opts).x)
    dt_g_my = (time.time() - t0) * 1000
    
    t0 = time.time()
    p_g_aas = np.asarray(equilibrium_profile(chem, T, pressures, element_vector_aas, Pref=1.0, options=opts).x)
    dt_g_aas = (time.time() - t0) * 1000
    
    # Plotting
    ax_pt, ax_vmr = axes[i, 0], axes[i, 1]
    ax_pt.plot(T, pressures, color="black", lw=2)
    ax_pt.set_yscale("log"); ax_pt.invert_yaxis(); ax_pt.set_xlim(0, 3000)
    ax_pt.set_xticks([0, 1000, 2000])
    ax_pt.set_ylabel("Pressure (bar)"); ax_pt.set_xlabel("Temperature (K)"); ax_pt.set_title(label)
    
    for sn, aliases, c in species_map:
        im = species_labels.index(sn)
        ig = next((chem.species.index(a) for a in aliases if a in chem.species), None)
        
        ax_vmr.plot(p_g_aas[:, ig], pressures, color=c, ls="-", lw=2, alpha=0.6)
        ax_vmr.plot(p_g_my[:, ig], pressures, color=c, ls=":", lw=2)
        ax_vmr.plot(p_ml[:, im], pressures, color=c, ls="--", lw=2)

    ax_vmr.set_xscale("log"); ax_vmr.set_xlim(1e-15, 3.0); ax_vmr.set_xlabel("Mixing Ratio")
    ax_vmr.set_title(f"ExoGibbs(AAS): {dt_g_aas:.1f}ms | ExoGibbs(Custom): {dt_g_my:.1f}ms | ML: {dt_ml:.1f}ms")
    ax_vmr.tick_params(axis='y', which='both', left=False, labelleft=False)
    
    if i == 0:
        h = [Line2D([0], [0], color='k', ls='-', alpha=0.6, label='ExoGibbs (AAS)'),
             Line2D([0], [0], color='k', ls=':', label='ExoGibbs (Custom)'),
             Line2D([0], [0], color='k', ls='--', label='ML')] + \
            [Line2D([0], [0], color=c, lw=3, label=n) for n, _, c in species_map]
        ax_vmr.legend(handles=h, loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10)

plt.show()